In [5]:
import pandas as pd
import ast
import os
import random
import M3  
import datetime
import nlp_techniques # Required for N-gram generation

# 1. Configuration
OUT_DIR = "../data/output_data"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR, exist_ok=True)

# M3 specific threshold from your notebook logic
string_matching_threshold = 0.3 

# 2. Data Import
RESEARCHER_SKILLS_PATH = "../data/input_data/Set_4/large_researcher_skills.csv"
PROPOSAL_SKILLS_PATH   = "../data/input_data/Set_4/large_proposal_skills.csv"
proposals_df = pd.read_csv(PROPOSAL_SKILLS_PATH) 
researchers_df = pd.read_csv(RESEARCHER_SKILLS_PATH) 

# 3. Helper for Name Normalization (Exactly as in User's M3 Notebook)
def normalize_name(name_str):
    return name_str.strip().replace(',', '')\
        .replace('.', '')\
        .replace('(', '')\
        .replace(')', '')\
        .replace(' ', '_')\
        .replace('-', '_')\
        .replace('\'', '_')\
        .lower()

# 4. M3 Skill Expansion (N-grams) & Preprocessing
m3_researcher_skills = {}
for i in range(len(researchers_df)):
    name = researchers_df.iloc[i]['researcher_name']
    skills_data = researchers_df.iloc[i]['skills']
    
    # Load skills as set/list
    skills = ast.literal_eval(skills_data)
    
    # Generate bigrams (2-grams)
    n_gram_interests = []
    for skill in skills:
        n_gram_interests.append(nlp_techniques.generate_N_grams(skill, ngram=2))
    
    # Flatten list of lists
    n_gram_interests = [item for sublist in n_gram_interests for item in sublist]
    
    # FIX: Use .union() to handle set + list addition
    m3_researcher_skills[name] = set(skills).union(n_gram_interests)

# 5. Parse Bandit Results (results_team.db)
pos_teams = {}
neg_teams = {}
RESULTS_PATH = "./boosted_results/test/results_team.db"

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, 'r') as file:
        lines = [line.rstrip() for line in file]
        for line in lines:
            if not line: continue
            
            # Identify Positive vs Negative association
            if line.startswith("team"):
                posFlag = True
                prefix_len = 5 # 'team('
            elif line.startswith("!team"):
                posFlag = False
                prefix_len = 6 # '!team('
            else:
                continue
                
            tokens = line.split(" ")
            if len(tokens) < 3: continue
            
            # Format: team(proposal, researcher) likelihood
            proposal_id = tokens[0][prefix_len:-1]
            researcher_norm = tokens[1][:-1]
            likelihood = float(tokens[2])
            
            target_dict = pos_teams if posFlag else neg_teams
            if proposal_id not in target_dict:
                target_dict[proposal_id] = {}
            target_dict[proposal_id][researcher_norm] = likelihood

# Mapping for bandit lookups
norm_to_real_name = {normalize_name(name): name for name in m3_researcher_skills.keys()}

def generate_m3_teams_boosted(pos_teams, neg_teams):
    results = []
    print("Start time:\t", datetime.datetime.now())

    for _, prop in proposals_df.iterrows():
        prop_link = prop['nsf_proposal_links_v0']
        # Extract proposal ID for Bandit matching
        prop_id = prop_link.split("/")[-1].replace('.htm', '')
        
        prop_skills = list(ast.literal_eval(prop['skills']))

        # Step 1: String Matching & Pseudo-Skill Filtering
        ranking, pseudo_skills = M3.string_matching_ranking(
            m3_researcher_skills, prop_skills, {}, matching_threshold=string_matching_threshold
        )

        # Step 2: Team Selection using Boosted Bandit findings
        team = []
        if prop_id in pos_teams:
            pos_members_norm = list(pos_teams[prop_id].keys())
            pos_members = [norm_to_real_name[m] for m in pos_members_norm if m in norm_to_real_name]
            if pos_members:
                team = random.sample(pos_members, min(len(pos_members), 4))
        
        # Fallback to Ranking and Negative Filtering
        if not team:
            lead = random.choice(list(m3_researcher_skills.keys()))
            m3_teams = M3.create_teams_for_each_person(ranking, lead, 1)
            if m3_teams:
                team = m3_teams[0]
                if prop_id in neg_teams:
                    neg_members_norm = list(neg_teams[prop_id].keys())
                    team = [m for m in team if normalize_name(m) not in neg_members_norm]

        # Step 4: Final Scoring with Ultra-Metric
        if team:
            goodness = M3.apply_ultra_metric(set(prop_skills), team, pseudo_skills)
            results.append({
                'nsf_proposal_links_v0': prop_link, 
                'lead_researcher': team[0],
                'team': team, 
                'goodness_score': goodness
            })

    print("End time:\t", datetime.datetime.now())
    return pd.DataFrame(results)

# 6. Execution and Saving to CSV
final_results_df = generate_m3_teams_boosted(pos_teams, neg_teams)
save_path = os.path.join(OUT_DIR, 'teaming_results_m3_final.csv')
final_results_df.to_csv(save_path, index=False)

print(f"Teaming complete. {len(final_results_df)} rows saved to {save_path}")

Start time:	 2026-02-10 16:52:49.223087
End time:	 2026-02-10 16:54:51.530953
Teaming complete. 500 rows saved to ../data/output_data/teaming_results_m3_final.csv
